# WiSig — Module 2 Final (single gate) + Sid scan

This notebook keeps the **single-threshold gating** logic unchanged:

- If `S_id < τ_id` ⇒ Unknown
- Else if `S_dom > τ_dom` ⇒ Known-Drift
- Else ⇒ Known-Stable

and only modifies **Sid**:
- `EmbMaha` (baseline)
- `Energy`
- `Fusion(EmbMaha + λ·Energy)` with several λ values

`S_dom` remains fixed as:
- **V1_mixNLL** (per-device × per-sourceRX mixture NLL)

We also increase the number of random TX splits (`TX_SPLIT_REPEATS`) so you can judge stability better.

In [2]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 10   # increased

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signal options ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- feature dims ----
D_DIM = 32

# ---- Sid scan ----
SID_FUSION_LAMBDAS = [0.25, 0.5, 1.0, 1.5, 2.0]

# ---- calibration targets ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# ---- output ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module2_sidscan/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, dataset_name=dataset_name, dataset_path=dataset_path,
    TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    SID_FUSION_LAMBDAS=SID_FUSION_LAMBDAS,
    FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_module2_sidscan/run_20260305_154022


In [3]:
def get_idx(lst, val): return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None: return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [4]:
# Choose subset + random SOURCE RXs (fixed by SEED)
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
SOURCE_RXS: ['1-1', '7-14', '7-7']
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']


In [5]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [6]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train: optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x/(np.linalg.norm(x,axis=axis,keepdims=True)+eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y==k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z,axis=1) @ P.T

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_EmbMaha(Z, mu, var):
    dif = Z[:,None,:] - mu[None,:,:]
    dist = np.sum((dif*dif)/(var[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore_from_ref(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

def build_sid_candidates(SidA_emb, SidA_energy, SidB_emb, SidB_energy, SidC_emb, SidC_energy,
                         SidD_emb, SidD_energy, SidE_emb, SidE_energy, SidF_emb, SidF_energy,
                         lambdas):
    # A=stable, B=drRX, C=drDAY, D=u_in, E=u_drRX, F=u_drDAY
    out = {}
    out["EmbMaha"] = dict(A=SidA_emb, B=SidB_emb, C=SidC_emb, D=SidD_emb, E=SidE_emb, F=SidF_emb)
    out["Energy"] = dict(A=SidA_energy, B=SidB_energy, C=SidC_energy, D=SidD_energy, E=SidE_energy, F=SidF_energy)

    zA_emb = zscore_from_ref(SidA_emb, SidA_emb)
    zB_emb = zscore_from_ref(SidB_emb, SidA_emb)
    zC_emb = zscore_from_ref(SidC_emb, SidA_emb)
    zD_emb = zscore_from_ref(SidD_emb, SidA_emb)
    zE_emb = zscore_from_ref(SidE_emb, SidA_emb)
    zF_emb = zscore_from_ref(SidF_emb, SidA_emb)

    zA_en = zscore_from_ref(SidA_energy, SidA_energy)
    zB_en = zscore_from_ref(SidB_energy, SidA_energy)
    zC_en = zscore_from_ref(SidC_energy, SidA_energy)
    zD_en = zscore_from_ref(SidD_energy, SidA_energy)
    zE_en = zscore_from_ref(SidE_energy, SidA_energy)
    zF_en = zscore_from_ref(SidF_energy, SidA_energy)

    for lam in lambdas:
        name = f"Fusion_lam{lam}"
        out[name] = dict(
            A=(zA_emb + lam*zA_en).astype(np.float32),
            B=(zB_emb + lam*zB_en).astype(np.float32),
            C=(zC_emb + lam*zC_en).astype(np.float32),
            D=(zD_emb + lam*zD_en).astype(np.float32),
            E=(zE_emb + lam*zE_en).astype(np.float32),
            F=(zF_emb + lam*zF_en).astype(np.float32),
        )
    return out

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

In [7]:
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list,0).astype(np.int64)
    return X,y,D,rx_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    return X,y,D

In [8]:
def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

In [9]:
def calibrate_thresholds(Sid_stable, Sdom_stable, frr_target=0.05, fdr_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_dom = float(np.quantile(Sdom_stable, 1.0 - fdr_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    # 0=Stable, 1=Drift, 2=Unknown
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def plot_gate_scatter(Sid, Sdom, gt, tau_id, tau_dom, out_png, title="", max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=gt[idx], alpha=0.5)
    plt.axvline(tau_id, linestyle="--")
    plt.axhline(tau_dom, linestyle="--")
    plt.xlabel("S_id (higher=known)")
    plt.ylabel("S_dom (higher=drift)")
    plt.title(title if title else "Single-threshold gate")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.yticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.xlabel("Pred"); plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

In [10]:
def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    X_drRX, _, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_drDAY, _, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_u_in, _, D_u_in = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
    X_u_drRX, _, D_u_drRX = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_u_drDAY, _, D_u_drDAY = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)

    rows=[]
    for fold,(idx_tr,idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1*len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val=-1.0; best_state=None; patience=0
        for ep in range(1,EPOCHS+1):
            _ = run_epoch(model, train_loader, optimizer=opt)
            _,va_acc = run_epoch(model, val_loader, optimizer=None)
            if va_acc > best_val + 1e-4:
                best_val=va_acc
                best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience=0
            else:
                patience += 1
                if patience >= PATIENCE: break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        model.load_state_dict(best_state)

        # stable A
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        acc = float((np.argmax(logits_A,1) == y_src[idx_te]).mean())
        D_A = D_src[idx_te]

        # train embeddings for prototypes & EmbMaha
        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        P = prototypes(Z_tr, y_src[idx_train], K)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)

        # eval embeddings/logits
        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        # Sid raw bases
        SidA_emb = sid_EmbMaha(Z_A, mu_z, var_z)
        SidB_emb = sid_EmbMaha(Z_B, mu_z, var_z)
        SidC_emb = sid_EmbMaha(Z_C, mu_z, var_z)
        SidD_emb = sid_EmbMaha(Z_D, mu_z, var_z)
        SidE_emb = sid_EmbMaha(Z_E, mu_z, var_z)
        SidF_emb = sid_EmbMaha(Z_F, mu_z, var_z)

        SidA_en = sid_Energy(logits_A)
        SidB_en = sid_Energy(logits_B)
        SidC_en = sid_Energy(logits_C)
        SidD_en = sid_Energy(logits_D)
        SidE_en = sid_Energy(logits_E)
        SidF_en = sid_Energy(logits_F)

        sid_candidates = build_sid_candidates(
            SidA_emb, SidA_en, SidB_emb, SidB_en, SidC_emb, SidC_en,
            SidD_emb, SidD_en, SidE_emb, SidE_en, SidF_emb, SidF_en,
            SID_FUSION_LAMBDAS
        )

        # fixed Sdom
        kA = np.argmax(cosine_to_proto(Z_A, P),1).astype(np.int64)
        kB = np.argmax(cosine_to_proto(Z_B, P),1).astype(np.int64)
        kC = np.argmax(cosine_to_proto(Z_C, P),1).astype(np.int64)
        kD = np.argmax(cosine_to_proto(Z_D, P),1).astype(np.int64)
        kE = np.argmax(cosine_to_proto(Z_E, P),1).astype(np.int64)
        kF = np.argmax(cosine_to_proto(Z_F, P),1).astype(np.int64)

        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        models_kr, fb = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)

        Sdom_A = sdom_V1_mixNLL(D_A, kA, models_kr, fb, source_rx_ids)
        Sdom_B = sdom_V1_mixNLL(D_drRX, kB, models_kr, fb, source_rx_ids)
        Sdom_C = sdom_V1_mixNLL(D_drDAY, kC, models_kr, fb, source_rx_ids)
        Sdom_D = sdom_V1_mixNLL(D_u_in, kD, models_kr, fb, source_rx_ids)
        Sdom_E = sdom_V1_mixNLL(D_u_drRX, kE, models_kr, fb, source_rx_ids)
        Sdom_F = sdom_V1_mixNLL(D_u_drDAY, kF, models_kr, fb, source_rx_ids)

        tau_dom = float(np.quantile(Sdom_A, 1.0 - FALSE_DRIFT_TARGET))

        # evaluate all Sid candidates under the SAME single gate
        sid_results = {}
        for sid_name, S in sid_candidates.items():
            tau_id = float(np.quantile(S["A"], FRR_TARGET))

            Sid_all = np.concatenate([S["A"], S["B"], S["C"], S["D"], S["E"], S["F"]])
            Sdom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D, Sdom_E, Sdom_F])
            gt = np.concatenate([
                np.zeros_like(S["A"], dtype=np.int64),
                np.ones_like(S["B"], dtype=np.int64),
                np.ones_like(S["C"], dtype=np.int64),
                np.full_like(S["D"], 2, dtype=np.int64),
                np.full_like(S["E"], 2, dtype=np.int64),
                np.full_like(S["F"], 2, dtype=np.int64),
            ])
            pred = gate_predict(Sid_all, Sdom_all, tau_id, tau_dom)
            cm = confusion_matrix(gt, pred, labels=[0,1,2])

            frr = float(1.0 - (pred[gt==0] == 0).mean())
            fdr = float((pred[gt==0] == 1).mean())
            far_unknown_accept = float((pred[gt==2] == 0).mean())
            miss_drift = float((pred[gt==1] == 0).mean())

            auc_unknown = roc_auc(
                np.concatenate([np.zeros_like(S["A"], dtype=np.int64), np.ones_like(S["D"], dtype=np.int64)]),
                np.concatenate([-S["A"], -S["D"]])
            )

            sid_results[sid_name] = dict(
                tau_id=tau_id,
                auc_unknown=auc_unknown,
                FRR_stable=frr,
                false_drift_stable=fdr,
                FAR_unknown_accept=far_unknown_accept,
                miss_drift=miss_drift,
                confusion_matrix=cm.tolist(),
            )

        # fixed score AUCs for this fold
        auc_drift_rx = roc_auc(
            np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_B, dtype=np.int64)]),
            np.concatenate([Sdom_A, Sdom_B])
        )
        auc_drift_day = roc_auc(
            np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_C, dtype=np.int64)]),
            np.concatenate([Sdom_A, Sdom_C])
        )

        # choose best Sid candidate by unknown AUC, tie-break on lower FAR_unknown_accept
        best_sid = None
        best_key = None
        for k,v in sid_results.items():
            score = (v["auc_unknown"], -v["FAR_unknown_accept"])
            if best_sid is None or score > best_sid:
                best_sid = score
                best_key = k

        # save plots for baseline and best
        for name in list(dict.fromkeys(["EmbMaha", best_key])):
            S = sid_candidates[name]
            tau_id = sid_results[name]["tau_id"]
            Sid_all = np.concatenate([S["A"], S["B"], S["C"], S["D"], S["E"], S["F"]])
            Sdom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D, Sdom_E, Sdom_F])
            gt = np.concatenate([
                np.zeros_like(S["A"], dtype=np.int64),
                np.ones_like(S["B"], dtype=np.int64),
                np.ones_like(S["C"], dtype=np.int64),
                np.full_like(S["D"], 2, dtype=np.int64),
                np.full_like(S["E"], 2, dtype=np.int64),
                np.full_like(S["F"], 2, dtype=np.int64),
            ])
            pred = gate_predict(Sid_all, Sdom_all, tau_id, tau_dom)
            cm = confusion_matrix(gt, pred, labels=[0,1,2])
            safe_name = name.replace(".", "p")
            plot_gate_scatter(Sid_all, Sdom_all, gt, tau_id, tau_dom, os.path.join(fold_dir, f"gate_scatter_{safe_name}.png"), title=f"{name} + single gate")
            plot_confmat(cm, os.path.join(fold_dir, f"gate_confmat_{safe_name}.png"), f"{name} + single gate")

        metrics = dict(
            split=split_id, fold=fold, acc=acc,
            auc_drift_rx=auc_drift_rx, auc_drift_day=auc_drift_day, tau_dom=tau_dom,
            sid_results=sid_results,
            best_sid=best_key,
        )
        with open(os.path.join(fold_dir, "metrics_sidscan.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        rows.append(metrics)
        print(
            f"[split {split_id} fold {fold}] acc={acc:.3f} "
            f"drRX={auc_drift_rx:.3f} | "
            f"EmbMaha AUCu={sid_results['EmbMaha']['auc_unknown']:.3f} FARunk={sid_results['EmbMaha']['FAR_unknown_accept']:.3f} | "
            f"best={best_key} AUCu={sid_results[best_key]['auc_unknown']:.3f} FARunk={sid_results[best_key]['FAR_unknown_accept']:.3f}"
        )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows=[]
for split_id in range(1, TX_SPLIT_REPEATS+1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)


=== TX split 1 ===
KNOWN_TX: ['20-15', '20-19', '14-10', '8-20']
UNKNOWN_TX: ['6-15', '14-7']
[split 1 fold 1] acc=0.999 drRX=0.950 | EmbMaha AUCu=0.785 FARunk=0.367 | best=EmbMaha AUCu=0.785 FARunk=0.367
[split 1 fold 2] acc=0.996 drRX=0.952 | EmbMaha AUCu=0.542 FARunk=0.402 | best=Energy AUCu=0.730 FARunk=0.410
[split 1 fold 3] acc=0.995 drRX=0.966 | EmbMaha AUCu=0.459 FARunk=0.160 | best=Fusion_lam1.0 AUCu=0.804 FARunk=0.160
[split 1 fold 4] acc=0.998 drRX=0.951 | EmbMaha AUCu=0.750 FARunk=0.335 | best=Fusion_lam0.5 AUCu=0.781 FARunk=0.368
[split 1 fold 5] acc=0.996 drRX=0.953 | EmbMaha AUCu=0.723 FARunk=0.388 | best=Fusion_lam0.5 AUCu=0.773 FARunk=0.387

=== TX split 2 ===
KNOWN_TX: ['8-20', '6-15', '20-15', '14-10']
UNKNOWN_TX: ['20-19', '14-7']
[split 2 fold 1] acc=0.996 drRX=0.922 | EmbMaha AUCu=0.997 FARunk=0.000 | best=EmbMaha AUCu=0.997 FARunk=0.000
[split 2 fold 2] acc=0.999 drRX=0.922 | EmbMaha AUCu=0.998 FARunk=0.000 | best=EmbMaha AUCu=0.998 FARunk=0.000
[split 2 fold 3]

In [11]:
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

summary = {}
summary["acc"] = dict(zip(["mean","std"], mean_std([r["acc"] for r in all_rows])))
summary["auc_drift_rx"] = dict(zip(["mean","std"], mean_std([r["auc_drift_rx"] for r in all_rows])))
summary["auc_drift_day"] = dict(zip(["mean","std"], mean_std([r["auc_drift_day"] for r in all_rows])))
summary["tau_dom"] = dict(zip(["mean","std"], mean_std([r["tau_dom"] for r in all_rows])))

sid_names = sorted({k for r in all_rows for k in r["sid_results"].keys()})
summary["sid"] = {}
for sid_name in sid_names:
    entry = {}
    for key in ["auc_unknown","FRR_stable","false_drift_stable","FAR_unknown_accept","miss_drift","tau_id"]:
        vals = [r["sid_results"][sid_name][key] for r in all_rows]
        m,s = mean_std(vals)
        entry[key] = {"mean": m, "std": s}
    cm_sum = np.zeros((3,3), dtype=np.int64)
    for r in all_rows:
        cm_sum += np.array(r["sid_results"][sid_name]["confusion_matrix"], dtype=np.int64)
    entry["confusion_matrix_sum"] = cm_sum.tolist()
    summary["sid"][sid_name] = entry

best_name_counts = {}
for r in all_rows:
    best_name_counts[r["best_sid"]] = best_name_counts.get(r["best_sid"], 0) + 1
summary["best_sid_counts"] = best_name_counts

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Global scores ===")
for key in ["acc","auc_drift_rx","auc_drift_day","tau_dom"]:
    print(f"{key:16s}: {summary[key]['mean']:.3f} ± {summary[key]['std']:.3f}")

print("\n=== Sid candidates (single gate) ===")
for sid_name in sid_names:
    s = summary["sid"][sid_name]
    print(f"\n[{sid_name}]")
    for key in ["auc_unknown","FRR_stable","false_drift_stable","FAR_unknown_accept","miss_drift","tau_id"]:
        print(f"{key:18s}: {s[key]['mean']:.3f} ± {s[key]['std']:.3f}")

print("\nBest Sid counts across splits×folds:")
print(best_name_counts)

=== Global scores ===
acc             : 0.997 ± 0.002
auc_drift_rx    : 0.936 ± 0.012
auc_drift_day   : 0.846 ± 0.018
tau_dom         : 2343.697 ± 282.746

=== Sid candidates (single gate) ===

[EmbMaha]
auc_unknown       : 0.917 ± 0.141
FRR_stable        : 0.093 ± 0.002
false_drift_stable: 0.043 ± 0.002
FAR_unknown_accept: 0.080 ± 0.136
miss_drift        : 0.239 ± 0.053
tau_id            : -1029.390 ± 128.349

[Energy]
auc_unknown       : 0.807 ± 0.114
FRR_stable        : 0.096 ± 0.002
false_drift_stable: 0.046 ± 0.002
FAR_unknown_accept: 0.106 ± 0.139
miss_drift        : 0.271 ± 0.050
tau_id            : 6.295 ± 1.016

[Fusion_lam0.25]
auc_unknown       : 0.928 ± 0.109
FRR_stable        : 0.093 ± 0.002
false_drift_stable: 0.043 ± 0.002
FAR_unknown_accept: 0.077 ± 0.138
miss_drift        : 0.231 ± 0.052
tau_id            : -1.005 ± 0.187

[Fusion_lam0.5]
auc_unknown       : 0.930 ± 0.103
FRR_stable        : 0.094 ± 0.002
false_drift_stable: 0.044 ± 0.002
FAR_unknown_accept: 0.078 ± 0.